# This Dataset Is Heavily Downsampled — and It Breaks Your Lag Features

**Playground Series S6E5 · Predicting F1 Pit Stops**

## TL;DR

Before you build any "previous lap" or "degradation trend" feature, check one thing:
**the lap records in this dataset are not continuous.**

- **~89%** of `(Race, Driver, Year)` groups are missing at least one lap.
- On average, **~30 laps are missing** per group — and an F1 race is only 50–70 laps.

That means each row should be treated as an **independent snapshot**, not a frame
in a continuous time series. Any feature that assumes "lap N−1 sits right before
lap N" is being computed on the wrong data.

This notebook verifies the problem in three short steps, and shows what to do instead.

## Setup

Load the training data and take a first look so we know what columns we have.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv('/kaggle/input/playground-series-s6e5/train.csv')

print('shape:', train.shape)
print('columns:', train.columns.tolist())
train.head()

## Step 1 — The assumption we're about to test

Intuitively, a single driver's race should produce one row per lap:
lap 1, lap 2, lap 3, ... all the way to the chequered flag.

If that holds, a lot of powerful features become available — last lap's pace,
tyre degradation trend, stint progression, rolling averages, and so on.

Let's check whether it actually holds. We group the data by driver, race and
season, then look at the lap numbers inside a few of those groups.

> Adjust `group_cols` and `'LapNumber'` below if your column names differ.

In [ ]:
group_cols = ['Race', 'Driver', 'Year']   # adjust to your column names
grouped = train.groupby(group_cols)

# Inspect the first few driver-races
for key, g in list(grouped)[:4]:
    laps = sorted(g['LapNumber'].tolist())
    full = set(range(min(laps), max(laps) + 1))
    missing = sorted(full - set(laps))
    print(f'Group {key}')
    print(f'  laps recorded : {len(laps):>3}  (from lap {min(laps)} to {max(laps)})')
    tail = ' ...' if len(missing) > 12 else ''
    print(f'  laps missing  : {len(missing):>3}  -> {missing[:12]}{tail}')
    print()

Already suspicious — several of these groups are missing laps in the
**middle**, not just at the start or end. But a handful of examples proves
nothing; maybe we just got unlucky. Let's check **every** group.

## Step 2 — How widespread is this?

For each `(Race, Driver, Year)` group, count how many laps are missing:

```
missing = (max lap − min lap + 1) − number of distinct laps recorded
```

If the data were continuous, this would be `0` for every single group.

In [ ]:
def missing_laps(g):
    laps = g['LapNumber']
    return (laps.max() - laps.min() + 1) - laps.nunique()

gaps = grouped.apply(missing_laps, include_groups=False)

print(f'Total driver-race groups       : {len(gaps):>8,}')
print(f'Groups with at least one gap   : {(gaps > 0).mean():>8.1%}')
print(f'Average laps missing per group : {gaps.mean():>8.1f}')
print()
print(gaps.describe())

The result is unambiguous:

- **~89%** of driver-race groups are missing at least one lap.
- On average **~30 laps are missing** per group; the median is even higher.

An F1 race is typically 50–70 laps. Losing ~30 of them means that most of the
time **we only see a fraction of each driver's race.** The data hasn't just
dropped a lap here and there — it has been heavily downsampled.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(gaps, bins=40, color='#E10600', edgecolor='white')
plt.axvline(gaps.mean(), color='black', linestyle='--',
            label=f'mean = {gaps.mean():.0f} laps')
plt.title('Missing laps per (Race, Driver, Year) group')
plt.xlabel('Number of missing laps')
plt.ylabel('Number of groups')
plt.legend()
plt.tight_layout()
plt.show()

The distribution makes it visual: only a small spike sits at zero — the
genuinely continuous groups. The overwhelming mass sits far to the right.
Most groups are missing many laps.

## Step 3 — What this means for feature engineering

If lap numbers aren't continuous, then "the row before" is **not** the previous
lap. The row before lap 40 might be lap 31. So any feature built from adjacent
rows is computed on the wrong data:

- Last lap's lap time / pace delta
- Tyre degradation *trend* over recent laps
- Rolling averages over a fixed lap window
- "Laps since last pit" derived by counting rows

These will silently produce numbers — no error, no warning — but the numbers
are meaningless. That is the dangerous kind of bug.

What is still **safe** are features that use a single row, or true per-group
aggregates that don't assume lap ordering:

- Within-row arithmetic: `TyreLife / Stint`, `TyreLife * RaceProgress`, ...
- One-hot encoding of categorical columns such as `Compound`
- Target encoding: e.g. each race's historical pit-stop rate
- Group-level statistics computed without relying on consecutive laps

## Conclusion

One cheap check — counting lap-number gaps — changed the entire feature
strategy for this competition:

1. The lap records are **downsampled**: ~89% of groups have gaps, ~30 laps
   missing on average.
2. Each row is best treated as an **independent snapshot**, not a time-series
   frame.
3. **Drop time-series / lag features**; build within-row and aggregate features
   instead.

EDA here didn't just describe the data — it closed off a dead end before any
time was wasted walking down it.

---

*If this saved you some debugging time, an upvote is appreciated. Spotted a flaw
in the reasoning? Let me know in the comments.*